# PyOccam Demo - Simple but Powerful OCCAM Analysis

This notebook demonstrates the core OCCAM workflow:

1. **Load data** - Initialize OCCAM with your dataset
2. **Search models** - Find the best models in the lattice
3. **Fit models** - Get detailed statistics for specific models
4. **Save results** - Export reports to files

**Run cells in order** for best results. You can modify parameters in the configuration cell.

## Configuration

Modify these settings before running the analysis:

In [1]:
# =============================================================================
# CONFIGURATION - Modify these as needed
# =============================================================================

import pyoccam
import os
from datetime import datetime

# Data settings
#DATA_FILE = "dementia05.txt"          # Your data file
pkg_dir = Path(pyoccam.__file__).parent
DATA_FILE = pkg_dir / "dementia05.txt"
# Search settings
SEARCH_TYPE = "loopless-up"           # loopless-up or full-up
SEARCH_LEVELS = 5                     # Search depth (3-7 recommended)
SEARCH_WIDTH = 3                      # Models to keep per level (3-5 recommended)

# Report settings
REPORT_VARIABLES = "Level$I, h, ddf, dLR, Alpha, %dH(DV), dAIC, dBIC"
SELECTION_CRITERION = "BIC"           # BIC, AIC, or Information
OUTPUT_FORMAT = "space"               # space, tab, or comma
SAVE_TO_FILES = True                  # Save results to files?

print("🚀 PyOccam Demo Configuration Loaded")
print(f"Data file: {DATA_FILE}")
print(f"Search: {SEARCH_TYPE}, {SEARCH_LEVELS} levels, width {SEARCH_WIDTH}")
print(f"Selection: {SELECTION_CRITERION}")

PyOccam 0.1.2 loaded successfully
PyOccam 0.1.2 loaded. Type pyoccam.help() for usage.


NameError: name 'Path' is not defined

## Step 1: Load Data

Initialize OCCAM with your dataset:

In [ ]:
# Check if data file exists
if not os.path.exists(DATA_FILE):
    print(f"❌ Error: Data file '{DATA_FILE}' not found!")
    print("Available files in current directory:")
    txt_files = [f for f in os.listdir('.') if f.endswith('.txt')]
    for f in txt_files[:10]:  # Show first 10
        print(f"  • {f}")
    if len(txt_files) > 10:
        print(f"  ... and {len(txt_files)-10} more")
    raise FileNotFoundError(f"Please set DATA_FILE to an existing file")

# Initialize OCCAM Manager
print(f"📊 Loading data: {DATA_FILE}")
print("-" * 50)

manager = pyoccam.VBMManager()
success = manager.init_from_command_line(["occam", DATA_FILE])

if not success:
    raise RuntimeError(f"Failed to load {DATA_FILE}. Check file format.")

print("✅ Data loaded successfully!")

# Display dataset information
print(f"\n📈 Dataset Information:")
print("-" * 50)
print(manager.get_basic_statistics())

variables = manager.get_variable_list()
print(f"Variables ({len(variables)}):")
for i, var in enumerate(variables, 1):
    print(f"  {i:2d}. {var}")
    
print(f"\nSample size: {manager.get_sample_size()}")
print(f"Has test data: {manager.has_test_data()}")

## Step 2: Configure and Run Search

Search for the best models in the lattice:

In [ ]:
print(f"🔍 Running Search Analysis")
print("-" * 50)
print(f"Algorithm: {SEARCH_TYPE}")
print(f"Levels: {SEARCH_LEVELS}")
print(f"Width: {SEARCH_WIDTH}")
print(f"Selection: {SELECTION_CRITERION}")

# Configure manager
manager.set_report_variables(REPORT_VARIABLES)
if OUTPUT_FORMAT == "comma":
    manager.set_report_separator(pyoccam.COMMASEP)
elif OUTPUT_FORMAT == "tab":
    manager.set_report_separator(pyoccam.TABSEP)
else:
    manager.set_report_separator(pyoccam.SPACESEP)

# Run search
print(f"\n⚡ Searching for best models...")
search_report = manager.generate_search_report(SEARCH_TYPE, SEARCH_LEVELS, SEARCH_WIDTH)

print(f"✅ Search complete!")

## Step 3: Examine Search Results

Display the best models found by different criteria:

In [ ]:
# Get best models by different criteria
best_bic = manager.get_best_model_by_bic()
best_aic = manager.get_best_model_by_aic()
best_info = manager.get_best_model_by_information()

print(f"🏆 Best Models Found:")
print("-" * 50)
print(f"  By BIC (Bayesian Information Criterion): {best_bic}")
print(f"  By AIC (Akaike Information Criterion): {best_aic}")
print(f"  By Information: {best_info}")

# Select model based on criterion
if SELECTION_CRITERION == "BIC":
    selected_model = best_bic
elif SELECTION_CRITERION == "AIC":
    selected_model = best_aic
else:
    selected_model = best_info

print(f"\n🎯 Selected model ({SELECTION_CRITERION}): {selected_model}")

# Save search results if requested
if SAVE_TO_FILES:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    search_filename = f"occam_search_{timestamp}.txt"
    
    print(f"\n💾 Saving search results to: {search_filename}")
    manager.write_report(search_filename)
    print(f"✅ Search report saved!")
else:
    print(f"\n💡 Set SAVE_TO_FILES = True to save search results to file")

## Step 4: Choose Model to Fit

You can fit the selected model or specify your own:

In [ ]:
# Model to fit - you can modify this cell to fit different models
# Examples:
#   fit_model = "IV"                    # Independence model
#   fit_model = "IV:ABC"                # Three-way interaction
#   fit_model = "IV:AB:AC:BC"           # All pairwise interactions
#   fit_model = selected_model          # Use the selected best model
#   fit_model = "IV:APOE:Gender"        # Custom model (adjust variable names)

fit_model = selected_model  # Use the selected model

print(f"🎛️  Model to Fit: {fit_model}")
print("-" * 50)
print(f"You can modify the 'fit_model' variable above to fit different models.")
print(f"")
print(f"Model syntax examples:")
print(f"  • IV                  - Independence model")
print(f"  • IV:ABC              - Three-way interaction")
print(f"  • IV:AB:AC:BC         - All pairwise interactions")
print(f"  • IV:Variable1        - Include specific variable")
print(f"")
print(f"Available variables: {', '.join(variables[:-1])}")
print(f"Target variable: {variables[-1]}")

## Step 5: Fit Model and Display Results

Generate detailed fit statistics for the chosen model:

In [ ]:
if fit_model:
    print(f"⚙️  Fitting Model: {fit_model}")
    print("-" * 50)
    
    try:
        # Generate fit report
        fit_report = manager.generate_fit_report(fit_model)
        print(f"✅ Model fit complete!")
        
        # Display basic fit information
        print(f"\n📊 Fit Results Summary:")
        print("-" * 50)
        print(f"Model: {fit_model}")
        print(f"The detailed fit report shows:")
        print(f"  • Model parameters and statistics")
        print(f"  • Contingency tables")
        print(f"  • Goodness-of-fit measures")
        print(f"  • Residual analysis")
        
        # Save fit report if requested
        if SAVE_TO_FILES:
            # Clean filename (remove invalid characters)
            clean_model = fit_model.replace(':', '_').replace(' ', '_')
            fit_filename = f"occam_fit_{clean_model}_{timestamp}.txt"
            fit_filename = "".join(c for c in fit_filename if c.isalnum() or c in "._-")
            
            print(f"\n💾 Saving fit report to: {fit_filename}")
            manager.write_fit_report(fit_filename)
            print(f"✅ Fit report saved!")
            
    except Exception as e:
        print(f"❌ Error fitting model '{fit_model}': {e}")
        print(f"")
        print(f"Common issues:")
        print(f"  • Check model syntax (e.g., 'IV:ABC' not 'IV ABC')")
        print(f"  • Ensure variable names match exactly")
        print(f"  • Model may be too complex for the data")
        print(f"")
        print(f"Available variables: {', '.join(variables)}")
else:
    print(f"No model specified for fitting.")
    print(f"Set fit_model to a valid model string in the cell above.")

## Step 6: Analysis Summary

Review what was accomplished and next steps:

In [ ]:
print(f"📋 Analysis Summary")
print("=" * 70)
print(f"Data file: {os.path.basename(DATA_FILE)}")
print(f"Sample size: {manager.get_sample_size()}")
print(f"Variables: {len(variables)}")
print(f"Search algorithm: {SEARCH_TYPE}")
print(f"Search levels: {SEARCH_LEVELS}")
print(f"Search width: {SEARCH_WIDTH}")
print(f"Selection criterion: {SELECTION_CRITERION}")
print(f"")
print(f"Best models found:")
print(f"  • BIC: {best_bic}")
print(f"  • AIC: {best_aic}")
print(f"  • Information: {best_info}")
print(f"")
print(f"Selected model: {selected_model}")
if 'fit_model' in locals() and fit_model:
    print(f"Fitted model: {fit_model}")

if SAVE_TO_FILES:
    print(f"\n📁 Files Generated:")
    if 'search_filename' in locals():
        print(f"  • Search report: {search_filename}")
    if 'fit_filename' in locals():
        print(f"  • Fit report: {fit_filename}")

print(f"\n✨ Next Steps:")
print(f"  1. Review the search report to understand the model space")
if 'fit_model' in locals() and fit_model:
    print(f"  2. Examine the fit report for detailed model statistics")
print(f"  3. Try different search parameters (levels, width, algorithm)")
print(f"  4. Fit and compare other models from the search results")
print(f"  5. Consider generating visualizations of the model structure")

print(f"\n🎉 Analysis complete! {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

## Advanced: Try Different Models

You can easily fit and compare multiple models:

In [ ]:
# Example: Compare multiple models
models_to_compare = [
    "IV",           # Independence model
    best_bic,       # Best by BIC
    best_aic,       # Best by AIC
    best_info       # Best by Information
]

# Remove duplicates while preserving order
models_to_compare = list(dict.fromkeys(models_to_compare))

print(f"🔄 Comparing Multiple Models")
print("=" * 70)

for i, model in enumerate(models_to_compare, 1):
    print(f"\n{i}. Fitting model: {model}")
    print("-" * 40)
    
    try:
        fit_report = manager.generate_fit_report(model)
        print(f"   ✅ Success")
        
        if SAVE_TO_FILES:
            clean_model = model.replace(':', '_').replace(' ', '_')
            compare_filename = f"occam_compare_{i}_{clean_model}_{timestamp}.txt"
            compare_filename = "".join(c for c in compare_filename if c.isalnum() or c in "._-")
            manager.write_fit_report(compare_filename)
            print(f"   💾 Saved to: {compare_filename}")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n🎯 Model comparison complete!")
print(f"Review the individual fit reports to compare model performance.")